### Spotify Listening Behavior Analysis and Skip Prediction

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [2]:
# Load data
df = pd.read_csv(
    r"C:\Users\hp\OneDrive\Desktop\data science\spotify_history.csv",
    encoding='latin1',
    low_memory=False
)
df

,ÿspotify_track_uri,ts,platform,ms_played,track_name,artist_name,album_name,reason_start,reason_end,shuffle,skipped
0,2J3n32GeLmMjwuAzyhcSNe,2013-07-08 02:44:34,web player,3185,"Say It, Just Say It",The Mowgli's,Waiting For The Dawn,autoplay,clickrow,False,False
1,1oHxIPqJyvAYHy0PVrDU98,2013-07-08 02:45:37,web player,61865,Drinking from the Bottle (feat. Tinie Tempah),Calvin Harris,18 Months,clickrow,clickrow,False,False
2,487OPlneJNni3NWC8SYqhW,2013-07-08 02:50:24,web player,285386,Born To Die,Lana Del Rey,Born To Die - The Paradise Edition,clickrow,unknown,False,False
3,5IyblF777jLZj1vGHG2UD3,2013-07-08 02:52:40,web player,134022,Off To The Races,Lana Del Rey,Born To Die - The Paradise Edition,trackdone,clickrow,False,False
4,0GgAAB0ZMllFhbNc3mAodO,2013-07-08 03:17:52,web player,0,Half Mast,Empire Of The Sun,Walking On A Dream,clickrow,nextbtn,False,False
...,...,...,...,...,...,...,...,...,...,...,...
149857,4Fz1WWr5o0OrlIcZxcyZtK,2024-12-15 23:06:19,android,1247,On The Way Home,John Mayer,Paradise Valley,fwdbtn,fwdbtn,True,True
149858,0qHMhBZqYb99yhX9BHcIkV,2024-12-15 23:06:21,android,1515,Magical Mystery Tour - Remastered 2009,The Beatles,Magical Mystery Tour,fwdbtn,fwdbtn,True,True
149859,0HHdujGjOZChTrl8lJWEIq,2024-12-15 23:06:22,android,1283,"Stop This Train - Live at the Nokia Theatre, L...",John Mayer,Where the Light Is: John Mayer Live In Los Ang...,fwdbtn,fwdbtn,True,True
149860,7peh6LUcdNPcMdrSH4JPsM,2024-12-15 23:06:23,android,1306,I Don't Trust Myself (With Loving You),John Mayer,Continuum,fwdbtn,fwdbtn,True,True


In [4]:
# Data cleaning
df['album_name'] = df['album_name'].fillna('Unknown Album')
df['reason_start'] = df['reason_start'].fillna('Unknown')
df['reason_end'] = df['reason_end'].fillna('Unknown')

In [5]:
df['shuffle'] = (
    df['shuffle']
    .astype(str)
    .str.strip()
    .str.lower()
    .map({'true': 1, 'false': 0})
    .fillna(0)
    .astype(int)
)

In [6]:
df.isnull().sum()

ÿspotify_track_uri    0
ts                    0
platform              0
ms_played             0
track_name            0
artist_name           0
album_name            0
reason_start          0
reason_end            0
shuffle               0
skipped               4
dtype: int64

### skipp prediction

In [7]:
df['skipped'] = (
    df['skipped']
    .astype(str)
    .str.strip()
    .str.lower()
    .map({'true': 1, 'false': 0, '1': 1, '0': 0})
)

In [56]:
df['skipped'] = pd.to_numeric(df['skipped'], errors='coerce')

In [57]:
# Convert to integer
df['skipped'] = df['skipped'].astype(int)

In [58]:
# 5. FEATURE ENGINEERING
# =========================
df['ts'] = pd.to_datetime(df['ts'], errors='coerce')

df['hour'] = df['ts'].dt.hour
df['day'] = df['ts'].dt.day_name()
df['month'] = df['ts'].dt.month
df['weekend'] = df['day'].isin(['Saturday', 'Sunday']).astype(int)

In [59]:
features = ['hour', 'ms_played', 'shuffle', 'weekend']

X = df[features]
y = df['skipped']

In [60]:
print("Target values:", y.unique())
print("Target type:", y.dtype)

Target values: [0 1]
Target type: int64


In [61]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [62]:
model = RandomForestClassifier()
model.fit(X_train, y_train)

RandomForestClassifier()

In [65]:
pred = model.predict(X_test)
pred

array([0, 0, 0, ..., 0, 0, 0])

In [66]:
print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.9231282530361671


### recommend songs

In [67]:
top_artists = df['artist_name'].value_counts().head(5)
print(top_artists)

artist_name
The Beatles       13621
The Killers        6878
John Mayer         4855
Bob Dylan          3814
Paul McCartney     2697
Name: count, dtype: int64


In [68]:
recommended = df[df['artist_name'].isin(top_artists.index)]

recommended_songs = recommended[['track_name','artist_name']].drop_duplicates()
recommended_songs.head(10)

,track_name,artist_name
22,Paper Doll,John Mayer
23,7th Street,John Mayer
24,"Baby, You've Got the Nonsense",John Mayer
25,Gravity - Radio Edit,John Mayer
26,Say,John Mayer
27,Waiting On the World to Change,John Mayer
28,I'm Gonna Find Another You - Acoustic,John Mayer
29,Daughters,John Mayer
30,Daughters - Electric Guitar Mix,John Mayer
31,"Free Fallin' - Live at the Nokia Theatre, Los ...",John Mayer


In [69]:
liked_songs = df[df['skipped'] == 0]

liked_songs[['track_name','artist_name']].drop_duplicates().head(10)

,track_name,artist_name
0,"Say It, Just Say It",The Mowgli's
1,Drinking from the Bottle (feat. Tinie Tempah),Calvin Harris
2,Born To Die,Lana Del Rey
3,Off To The Races,Lana Del Rey
4,Half Mast,Empire Of The Sun
5,Impossible,James Arthur
6,We Own The Sky,M83
7,Higher Ground - Remastered 2003,Red Hot Chili Peppers
8,Happy Up Here,Röyksopp
9,Phantom,Justice
